In [1]:
from pathlib import Path
import sys
import pandas as pd
import tensorflow as tf
import mlflow
import mlflow.tensorflow

PROJECT_ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

from src.data.cityscapes_labels import GROUPS, G
from src.data.cityscapes_tfdata import make_cityscapes_ds
from src.models.unet import build_unet
from src.models.metrics import MeanIoUIgnoreVoid

print("Project root:", PROJECT_ROOT)
print("Groups:", GROUPS, "void_id:", G["void"])

2026-03-09 13:27:59.833413: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-09 13:27:59.838983: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-09 13:28:00.146913: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-09 13:28:01.465786: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

Project root: /home/aurelien/formation_openclassrooms/projet_8/cityseg-project
Groups: ['flat', 'human', 'vehicle', 'construction', 'object', 'nature', 'sky', 'void'] void_id: 7


In [2]:
CSV_PATH = PROJECT_ROOT / "data/manifests/cityscapes_pairs.csv"
df = pd.read_csv(CSV_PATH)

ROOT = PROJECT_ROOT / "data"
df["img_abs"]  = df["image_path"].apply(lambda p: str(ROOT / p))
df["mask_abs"] = df["mask_path"].apply(lambda p: str(ROOT / p))

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)

TARGET_HW = (256, 512)
BATCH = 4

ds_train = make_cityscapes_ds(
    train_df["img_abs"].tolist(),
    train_df["mask_abs"].tolist(),
    target_hw=TARGET_HW,
    batch_size=BATCH,
    training=True,
    use_augmentation=False,
    cache=False,
)

ds_val = make_cityscapes_ds(
    val_df["img_abs"].tolist(),
    val_df["mask_abs"].tolist(),
    target_hw=TARGET_HW,
    batch_size=BATCH,
    training=False,
    use_augmentation=False,
    cache=False,
)

E0000 00:00:1773059288.894341   49067 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1773059288.898638   49067 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
NUM_CLASSES = len(GROUPS)
VOID_ID = G["void"]

model = build_unet(input_shape=(TARGET_HW[0], TARGET_HW[1], 3), num_classes=NUM_CLASSES, base_filters=32, dropout=0.1)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

miou = MeanIoUIgnoreVoid(num_classes=NUM_CLASSES, void_id=VOID_ID)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=[miou],
)

model.summary()

Model: "unet_baseline"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 512,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 512,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 256, 512,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 512,  │      9,216 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 512,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 256, 512,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 256,  │          0 │ re_lu_1[0][0]     │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 256,  │     18,432 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 256,  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 128, 256,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 256,  │     36,864 │ re_lu_2[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 256,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 128, 256,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 128,   │          0 │ re_lu_3[0][0]     │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 128,   │     73,728 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 128,   │        512 │ conv2d_4[0][0]  

 Total params: 7,855,720 (29.97 MB)

 Trainable params: 7,849,832 (29.94 MB)

 Non-trainable params: 5,888 (23.00 KB)

In [4]:
EXPERIMENT_NAME = "cityseg-unet"
mlflow.set_tracking_uri(f"file:{PROJECT_ROOT}/mlruns")
mlflow.set_experiment(EXPERIMENT_NAME)

EPOCHS = 10

# callbacks utiles
cb = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(PROJECT_ROOT / "models/unet_full_aug_best.keras"),
        monitor="val_miou_no_void",
        mode="max",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(monitor="val_miou_no_void", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_miou_no_void", mode="max", factor=0.5, patience=2, min_lr=1e-6),
]

with mlflow.start_run(run_name="unet_full_aug_256x512"):
    # params
    mlflow.log_params({
        "model": "unet",
        "target_h": TARGET_HW[0],
        "target_w": TARGET_HW[1],
        "batch": BATCH,
        "lr": 1e-3,
        "epochs": EPOCHS,
        "void_ignored": True,
        "use_augmentation": False,
    })

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS,
        callbacks=cb,
    )

    # log best metrics (simple)
    best_val = max(history.history.get("val_miou_no_void", [0.0]))
    mlflow.log_metric("best_val_miou_no_void", float(best_val))

    # save + log model (keras format)
    out_model = PROJECT_ROOT / "models/unet_full_aug_final.keras"
    out_model.parent.mkdir(parents=True, exist_ok=True)
    model.save(out_model)

    mlflow.log_artifact(str(out_model), artifact_path="model")

Epoch 1/10


/home/aurelien/formation_openclassrooms/projet_8/cityseg-project/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


743/743 ━━━━━━━━━━━━━━━━━━━━ 976s 1s/step - loss: 0.4472 - miou_no_void: 0.5022 - val_loss: 0.4979 - val_miou_no_void: 0.5065 - learning_rate: 0.0010
Epoch 2/10
743/743 ━━━━━━━━━━━━━━━━━━━━ 1005s 1s/step - loss: 0.3032 - miou_no_void: 0.5979 - val_loss: 0.3491 - val_miou_no_void: 0.5814 - learning_rate: 0.0010
Epoch 3/10
743/743 ━━━━━━━━━━━━━━━━━━━━ 977s 1s/step - loss: 0.2585 - miou_no_void: 0.6434 - val_loss: 0.3166 - val_miou_no_void: 0.6134 - learning_rate: 0.0010
Epoch 4/10
743/743 ━━━━━━━━━━━━━━━━━━━━ 979s 1s/step - loss: 0.2278 - miou_no_void: 0.6779 - val_loss: 0.2395 - val_miou_no_void: 0.6734 - learning_rate: 0.0010
Epoch 5/10


2026-03-09 14:33:57.557364: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


743/743 ━━━━━━━━━━━━━━━━━━━━ 974s 1s/step - loss: 0.2084 - miou_no_void: 0.7004 - val_loss: 0.2647 - val_miou_no_void: 0.6450 - learning_rate: 0.0010
Epoch 6/10
743/743 ━━━━━━━━━━━━━━━━━━━━ 978s 1s/step - loss: 0.1925 - miou_no_void: 0.7187 - val_loss: 0.2232 - val_miou_no_void: 0.7013 - learning_rate: 0.0010
Epoch 7/10
743/743 ━━━━━━━━━━━━━━━━━━━━ 974s 1s/step - loss: 0.1784 - miou_no_void: 0.7344 - val_loss: 0.2250 - val_miou_no_void: 0.6920 - learning_rate: 0.0010
Epoch 8/10


...

743/743 ━━━━━━━━━━━━━━━━━━━━ 979s 1s/step - loss: 0.1733 - miou_no_void: 0.7421 - val_loss: 0.1858 - val_miou_no_void: 0.7310 - learning_rate: 0.0010
Epoch 9/10
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.1632 - miou_no_void: 0.7548

2026-03-09 15:54:38.702091: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


743/743 ━━━━━━━━━━━━━━━━━━━━ 976s 1s/step - loss: 0.1585 - miou_no_void: 0.7595 - val_loss: 0.2060 - val_miou_no_void: 0.7137 - learning_rate: 0.0010
Epoch 10/10


2026-03-09 15:55:18.413061: W tensorflow/core/kernels/data/prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 10485760 bytes after encountering the first element of size 10485760 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


743/743 ━━━━━━━━━━━━━━━━━━━━ 978s 1s/step - loss: 0.1559 - miou_no_void: 0.7635 - val_loss: 0.1892 - val_miou_no_void: 0.7310 - learning_rate: 0.0010
